# Livestock numbers

This notebook generates a dataset containing the latest livestock data numbers
for each local authority in the UK, by combining officially reported numbers
from each country. Each dataset is formatted and then they are concatenated into
a single xarray DataArray with the following coordinates:

- **Code**: LAD GSS Code 
- Name: LAD official name (used as labelling coordinate)
- **Species** (Pigs, sheep, cattle, poultry)


Original data is obtained from the following sources:

### England
Local Authority District data can be found in the [June agricultural survey data page](https://www.gov.uk/government/statistical-data-sets/structure-of-the-agricultural-industry-in-england-and-the-uk-at-june).
Farm holdings, area and livestock numbers are reported for various
administrative units. We are interested in "Local authority" numbers.

### Scotland
Breakdown of crops and livestock data is available in the June Agricultural
survey [Ad hoc requests page of the scottish government](https://www.gov.scot/publications/june-agricultural-census-ad-hoc-requests/pages/livestock/)
under "Breakdown of crops and livestock by local authority: 2005 to 2025".

### Northern Ireland
June Agricultural survey data, including livestock numbers by district council
are available from the [DAERA website](https://www.daera-ni.gov.uk/publications/agricultural-census-northern-ireland-2025)

### Wales
Data and statistics for small areas is only available up to 2020. More recent
data has been removed following the identification of disclosive figures for
some farm categories.
We will be using the [last available dataset for agricultural small area
statistics](https://www.gov.wales/agricultural-small-area-statistics-2002-2020)

### LAD names and codes
Local Authority District names and codes are avaiable from the [ONS Geography
Portal](https://geoportal.statistics.gov.uk/) by searching "Local Authority
Districts Names and Codes in the UK" 


In [1]:
import pandas as pd
import xarray as xr
from matplotlib import pyplot as plt

In [2]:
LAD_name_fname = "Local_Authority_Districts_(April_2025)_Names_and_Codes_in_the_UK_v2.csv"

eng_data_fname = "structure-england-june-local-authority-16apr26.ods"
sco_data_fname = "Crops+and+livestock+by+local+authority+2005+to+2025.xlsx"
nir_data_fname = "Agricultural Census in Northern Ireland 2025 - Data Tables.ods"
wal_data_fname = "agricultural-small-area-statistics-2002-2020-tables-637.ods"


In [3]:
lad_names = pd.read_csv(LAD_name_fname)
lad_names.head()

,LAD25CD,LAD25NM,LAD25NMW,ObjectId
0,E06000001,Hartlepool,NaN,1
1,E06000002,Middlesbrough,NaN,2
2,E06000003,Redcar and Cleveland,NaN,3
3,E06000004,Stockton-on-Tees,NaN,4
4,E06000005,Darlington,NaN,5


## England

In [4]:
cols = [
    "Local Authority(2)",
    "Cattle and Calves(4)",
    "Sheep and lambs",
    "Pigs",
    "Poultry"
    ]

eng_data = pd.read_excel(eng_data_fname, sheet_name="2025", skiprows=2, nrows=305, usecols=cols)

# Rename columns for uniformity and clarity
eng_data.columns = ["Name", "Cattle", "Sheep", "Pigs", "Poultry"]

eng_data.tail()

,Name,Cattle,Sheep,Pigs,Poultry
300,Gloucester,#,843.972225,#,#
301,Stroud,30148,35128.930153,2389.149928,177412.356975
302,Tewkesbury,17521,59223.424639,3497.260539,902465.596084
303,South West,1556857,2664467.026989,336609.7002,15628253.567535
304,England,4660871,13312071.000012,3653771.000002,133035903.999861


In [5]:
# normalize and map
lad_map = lad_names.assign(
    LAD25NM = lad_names['LAD25NM'].astype(str).str.strip().str.upper(),
    LAD25CD = lad_names['LAD25CD'].astype(str).str.strip()
).set_index('LAD25NM')['LAD25CD']

eng_data['Code'] = (
    eng_data['Name'].astype(str).str.strip().str.upper().map(lad_map)
)

# Missing data in the English dataset is represented by a "#" character, so we will replace those values with zero for consistency.
eng_data.replace("#", 0, inplace=True)

eng_data.head()

,Name,Cattle,Sheep,Pigs,Poultry,Code
0,Hartlepool,2780,5920.182208,4559.179441,569497.712197,E06000001
1,Middlesbrough,0,0,0,0,E06000002
2,Redcar and Cleveland,6507,8958.250507,15318.494141,2904.991213,E06000003
3,Stockton-on-Tees,2521,8627.058972,9240.791436,3857.928406,E06000004
4,Darlington,9999,17497.864602,12463.753792,508116.256567,E06000005


## Scotland

In [6]:
cols = [
    "Local authority",
    "Total cattle (Number)",
    "Total sheep (Number)",
    "Total pigs (Number)",
    "Total poultry (Number)"
    ]

sco_data = pd.read_excel(sco_data_fname, sheet_name="Table_2", skiprows=5, nrows=32, usecols=cols)

# Rename columns for uniformity and clarity
sco_data.columns = ["Name", "Cattle", "Sheep", "Pigs", "Poultry"]

sco_data.tail()

,Name,Cattle,Sheep,Pigs,Poultry
27,South Ayrshire,75084.0,206002,243,43917
28,South Lanarkshire,80085.0,301872,4735,180729
29,Stirling,35965.0,197096,993,32163
30,West Dunbartonshire,3193,14706,c,206
31,West Lothian,16546.0,47787,1349,1072719


In [7]:
# Some Local Authority names in the Scottish data contain a "&" character, which is not present in the LAD names.
# We will replace "&" with "and" to ensure consistency.
sco_data['Name'] = sco_data['Name'].str.replace('&', 'and', regex=False)

sco_data['Code'] = (
    sco_data['Name'].astype(str).str.strip().str.upper().map(lad_map)
)

# Missing data in the Scottish dataset is represented by a "c" character, so we will replace those values with zero for consistency.
sco_data.replace("c", 0, inplace=True)

sco_data.head()

,Name,Cattle,Sheep,Pigs,Poultry,Code
0,Aberdeen City,5851.0,4588,88,956,S12000033
1,Aberdeenshire,251410.999999,487303,147612,1395547,S12000034
2,Angus,41511.0,130250,15083,1010066,S12000041
3,Argyll and Bute,57280.0,413008,1141,6363,S12000035
4,City of Edinburgh,0,22410,0,623949,S12000036


## Northern Ireland

In [8]:
cols = [
    "District Council",
    "Total Cattle",
    "Total Sheep",
    "Total Pigs",
    "Total Poultry"
    ]

nir_data = pd.read_excel(nir_data_fname, sheet_name="Table_6_6", skiprows=15, nrows=11, usecols=cols, thousands=',')

# Rename columns for uniformity and clarity
nir_data.columns = ["Name", "Cattle", "Sheep", "Pigs", "Poultry"]

nir_data.head()

,Name,Cattle,Sheep,Pigs,Poultry
0,Antrim and Newtownabbey,78307,74793,22178,875096
1,"Armagh City, Banbridge and Craigavon",253300,103583,259661,3608164
2,Belfast [1],3461,3498,25,20
3,Causeway Coast and Glens,192041,370322,8000,3061249
4,Derry City and Strabane,112638,249069,43202,562748


In [9]:
# Belfast has a note indicator "[1]" which we will remove to ensure consistency with the LAD names.
nir_data['Name'] = nir_data['Name'].str.replace(r'\[1\]', '', regex=True).str.strip()

nir_data['Code'] = (
    nir_data['Name'].astype(str).str.strip().str.upper().map(lad_map)
)

nir_data

,Name,Cattle,Sheep,Pigs,Poultry,Code
0,Antrim and Newtownabbey,78307,74793,22178,875096,N09000001
1,"Armagh City, Banbridge and Craigavon",253300,103583,259661,3608164,N09000002
2,Belfast,3461,3498,25,20,N09000003
3,Causeway Coast and Glens,192041,370322,8000,3061249,N09000004
4,Derry City and Strabane,112638,249069,43202,562748,N09000005
5,Fermanagh and Omagh,276756,257616,52568,3133170,N09000006
6,Lisburn and Castlereagh,68171,35672,17144,701777,N09000007
7,Mid and East Antrim,122607,240480,16743,3487957,N09000008
8,Mid Ulster,268736,206285,242578,8618895,N09000009
9,"Newry, Mourne and Down",209143,243873,67530,1572008,N09000010


## Wales

In [10]:
cols = [
    "Year",
    "Name",
    "Total sheep",
    "Total cattle (CTS)",
    "Pigs",
    "Poultry"
    ]

wal_data = pd.read_excel(wal_data_fname, sheet_name="SmallAreas", skiprows=4, usecols=cols)

# Rename columns for uniformity and clarity
wal_data.columns = ["Year", "Name", "Sheep", "Cattle", "Pigs", "Poultry"]
wal_data = wal_data[wal_data['Year'] == 2020]  # Filter for the year 2020


# Missing data in the Wales dataset is represented by an "X" character, so we will replace those values with zero for consistency.
wal_data.replace("X", 0, inplace=True)

wal_data.head()

,Year,Name,Sheep,Cattle,Pigs,Poultry
4270,2020,Residual,0,0,20364,5181172
4271,2020,ISLE01,28737,3056,43,281
4272,2020,ISLE02,23590,5907,0,63290
4273,2020,ISLE03,28326,7801,0,620
4274,2020,ISLE04,40449,7532,0,0


In [11]:
# Wales dataset contains a lookup table for Local Authority names and codes, so we will use that to map the names to codes.
wal_lad_map = pd.read_excel(wal_data_fname, sheet_name="Lookup", usecols=["Name", "Code"], skiprows=17, nrows=22)
wal_lad_map

,Code,Name
0,ISLE,Isle of Anglesey
1,GWYN,Gwynedd
2,CONW,Conwy
3,DENB,Denbighshire
4,FLIN,Flintshire
5,WREX,Wrexham
6,POWY,Powys
7,CERE,Ceredigion
8,PEMB,Pembrokeshire
9,CARM,Carmarthenshire


In [12]:
# Additionally, each Local Authority District is subdivided into smaller areas.
# The dataset contains a mapping of these smaller areas to their corresponding Local Authority Districts.
# We will use this mapping to ensure that we have the correct Local Authority Codes for each entry in the dataset.

wal_small_lad_map = pd.read_excel(wal_data_fname, sheet_name="Lookup", usecols=["Code", "Local authority"], skiprows=43, nrows=235)

wal_small_lad_map["Name"] = (
    wal_small_lad_map['Local authority'].astype(str).str.strip().str.upper().map(
        wal_lad_map.set_index('Code')['Name']
    )
)
wal_small_lad_map.head()

,Code,Local authority,Name
0,ISLE01,ISLE,Isle of Anglesey
1,ISLE02,ISLE,Isle of Anglesey
2,ISLE03,ISLE,Isle of Anglesey
3,ISLE04,ISLE,Isle of Anglesey
4,ISLE05,ISLE,Isle of Anglesey


In [13]:
# Now we map the Local Authority IDs in the main dataset to their corresponding
# names using the mapping from above.
wal_data["Name"] = (
    wal_data['Name'].astype(str).str.strip().str.upper().map(
        wal_small_lad_map.set_index('Code')['Name']
    )
)
wal_data

,Year,Name,Sheep,Cattle,Pigs,Poultry
4270,2020,NaN,0,0,20364,5181172
4271,2020,Isle of Anglesey,28737,3056,43,281
4272,2020,Isle of Anglesey,23590,5907,0,63290
4273,2020,Isle of Anglesey,28326,7801,0,620
4274,2020,Isle of Anglesey,40449,7532,0,0
...,...,...,...,...,...,...
4502,2020,Monmouthshire,24791,4717,433,0
4503,2020,Monmouthshire,6451,2633,35,649
4504,2020,Monmouthshire,9713,3607,0,0
4505,2020,Monmouthshire,10296,3935,19,0


In [14]:
# Next, we sum over all the smaller areas to get the total counts for each Local Authority District in Wales.
wal_data_grouped = wal_data.groupby("Name").agg({
    "Cattle": "sum",
    "Sheep": "sum",
    "Pigs": "sum",
    "Poultry": "sum"
}).reset_index()

# Rhondda Cynon Taf is represented as "Rhondda Cynon Taff" in the Wales dataset, so we will correct that to ensure consistency with the LAD names.
wal_data_grouped['Name'] = wal_data_grouped['Name'].replace("Rhondda Cynon Taff", "Rhondda Cynon Taf")

# Finally, we add the Local Authority Codes to the grouped dataset.
wal_data_grouped["Code"] = (
    wal_data_grouped['Name'].astype(str).str.strip().str.upper().map(lad_map)
)
wal_data_grouped

,Name,Cattle,Sheep,Pigs,Poultry,Code
0,Blaenau Gwent,609,26917,86,803,W06000019
1,Bridgend,6732,65041,298,758,W06000013
2,Caerphilly,7551,75580,125,9251,W06000018
3,Cardiff,1715,9812,254,426,W06000015
4,Carmarthenshire,191648,738460,908,14271,W06000010
5,Ceredigion,111356,794419,522,11038,W06000008
6,Conwy,55913,737824,308,3594,W06000003
7,Denbighshire,53432,454853,185,2182,W06000004
8,Flintshire,34651,95931,96,4225,W06000005
9,Gwynedd,85190,1096865,913,11940,W06000002


## Concatenating

In [15]:
# Finally, we combine all the datasets into a single DataFrame for further analysis.
combined_data = pd.concat([eng_data, sco_data, nir_data, wal_data_grouped], ignore_index=True)


# Remove all rows with missing Local Authority Codes
combined_data = combined_data.dropna(subset=['Code'])

# Convert all data to integer type for consistency
combined_data[['Cattle', 'Sheep', 'Pigs', 'Poultry']] = combined_data[['Cattle', 'Sheep', 'Pigs', 'Poultry']].astype(int)

combined_data

,Name,Cattle,Sheep,Pigs,Poultry,Code
0,Hartlepool,2780,5920,4559,569497,E06000001
1,Middlesbrough,0,0,0,0,E06000002
2,Redcar and Cleveland,6507,8958,15318,2904,E06000003
3,Stockton-on-Tees,2521,8627,9240,3857,E06000004
4,Darlington,9999,17497,12463,508116,E06000005
...,...,...,...,...,...,...
365,Rhondda Cynon Taf,5649,122286,167,3679,W06000016
366,Swansea,11888,79961,155,10097,W06000011
367,Torfaen,1576,12287,85,621,W06000020
368,Vale of glamorgan,17469,58369,333,1158,W06000014


In [16]:
combined_data_indexed = combined_data.set_index(['Code'])
combined_data_ds = combined_data_indexed.to_xarray().set_coords(["Name"])

combined_data_da = combined_data_ds.to_array(dim="Species", name="Livestock counts")
combined_data_da

<xarray.DataArray 'Livestock counts' (Species: 4, Code: 361)> Size: 12kB
array([[  2780,      0,   6507, ...,   1576,  17469,  39422],
       [  5920,      0,   8958, ...,  12287,  58369, 117453],
       [  4559,      0,  15318, ...,     85,    333,    714],
       [569497,      0,   2904, ...,    621,   1158,   6899]],
      shape=(4, 361))
Coordinates:
  * Species  (Species) object 32B 'Cattle' 'Sheep' 'Pigs' 'Poultry'
  * Code     (Code) object 3kB 'E06000001' 'E06000002' ... 'W06000006'
    Name     (Code) object 3kB 'Hartlepool' 'Middlesbrough' ... 'Wrexham'

In [17]:
# Save as a NetCDF.
combined_data_da.to_netcdf("../data/UK_LIVESTOCK_LAD.nc")